In [21]:
import pandas as pd

df = pd.read_csv('../data/raw/Sample - Superstore.csv',encoding='latin-1')

#convertir fechas de string a datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

print("tipos de datos despues de la conversion: ")
print(df[['Order Date','Ship Date']].dtypes)

tipos de datos despues de la conversion: 
Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object


In [22]:
#Extraer componentes de fecha
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Quarter'] = df['Order Date'].dt.quarter

#Calcular dias entre orden y envio
df["Days to Ship"] = (df["Ship Date"] - df['Order Date']).dt.days

print("Nuevas columnas creadas: ")
print(df[['Order Date','Ship Date','Order Year','Order Month','Order Quarter','Days to Ship']].head())



Nuevas columnas creadas: 
  Order Date  Ship Date  Order Year  Order Month  Order Quarter  Days to Ship
0 2016-11-08 2016-11-11        2016           11              4             3
1 2016-11-08 2016-11-11        2016           11              4             3
2 2016-06-12 2016-06-16        2016            6              2             4
3 2015-10-11 2015-10-18        2015           10              4             7
4 2015-10-11 2015-10-18        2015           10              4             7


In [23]:
#Calcular margen porcentual
df['Profit Margin %'] = round((df['Profit'] / df['Sales']) * 100, 2)

print("Estadisticas de Profit Margin %: ")
print(df['Profit Margin %'].describe())

Estadisticas de Profit Margin %: 
count    9994.000000
mean       12.031390
std        46.675436
min      -275.000000
25%         7.500000
50%        27.000000
75%        36.250000
max        50.000000
Name: Profit Margin %, dtype: float64


In [24]:
duplicados = df.duplicated().sum()
print(f'Filas duplicados: {duplicados}')

#Verificar que Order ID + Product ID no se repitan
duplicados_orden = df.duplicated(subset=['Order ID','Product ID']).sum()
print(f'Combinaciones Order ID + Product ID duplicadas: {duplicados_orden}')

Filas duplicados: 0
Combinaciones Order ID + Product ID duplicadas: 8


0 filas completamente duplicadas → no hay registros idénticos, el dato está limpio

8 combinaciones Order ID + Product ID duplicadas → hay 8 casos donde el mismo producto aparece dos veces en la misma orden

In [25]:
#Ver los casos duplicados
mascara = df.duplicated(subset=['Order ID','Product ID'], keep=False)
duplicados_detalle = df[mascara].sort_values(['Order ID','Product ID'])
print(duplicados_detalle[["Order ID",'Product ID','Sales','Quantity','Discount','Profit']])

            Order ID       Product ID     Sales  Quantity  Discount    Profit
6498  CA-2015-103135  OFF-BI-10000069   135.090         9       0.0   62.1414
6500  CA-2015-103135  OFF-BI-10000069    90.060         6       0.0   41.4276
350   CA-2016-129714  OFF-PA-10001970    24.560         2       0.0   11.5432
352   CA-2016-129714  OFF-PA-10001970    49.120         4       0.0   23.0864
1300  CA-2016-137043  FUR-FU-10003664   572.760         6       0.0  166.1004
1301  CA-2016-137043  FUR-FU-10003664   286.380         3       0.0   83.0502
9168  CA-2016-140571  OFF-PA-10001954   319.760        14       0.0  147.0896
9169  CA-2016-140571  OFF-PA-10001954    45.680         2       0.0   21.0128
7881  CA-2017-118017  TEC-AC-10002006    76.752         6       0.2   10.5534
7882  CA-2017-118017  TEC-AC-10002006   102.336         8       0.2   14.0712
3183  CA-2017-152912  OFF-ST-10003208  1633.140         9       0.0  473.6106
3184  CA-2017-152912  OFF-ST-10003208   544.380         3       

Eso responde la pregunta.
Son registros legítimos, no errores. El mismo producto en la misma orden pero con cantidades distintas, probablemente porque se procesaron en dos envíos separados. No hay que eliminarlos.

In [26]:
# Tabla de Clientes
dim_cliente = df[['Customer ID','Customer Name','Segment']].drop_duplicates()
dim_cliente.columns = ['customer_id','customer_name','segment']

# Tabla de productos
dim_producto = df[['Product ID', 'Product Name', 'Category', 'Sub-Category']].drop_duplicates(subset=['Product ID'], keep='first')
dim_producto.columns = ['product_id', 'product_name', 'category', 'sub_category']

print(f"Productos únicos: {len(dim_producto)}")
print(f"Duplicados restantes: {dim_producto['product_id'].duplicated().sum()}")

# Tabla de ubicación
dim_ubicacion = df[['City', 'State', 'Country', 'Region', 'Postal Code']].drop_duplicates()
dim_ubicacion = dim_ubicacion.reset_index(drop=True)
dim_ubicacion['location_id'] = dim_ubicacion.index + 1
dim_ubicacion.columns = ['city', 'state', 'country', 'region', 'postal_code', 'location_id']

print(f"Clientes: {len(dim_cliente)} registros")
print(f"Productos: {len(dim_producto)} registros")
print(f"Ubicaciones: {len(dim_ubicacion)} registros")

Productos únicos: 1862
Duplicados restantes: 0
Clientes: 793 registros
Productos: 1862 registros
Ubicaciones: 632 registros


In [27]:
print(dim_producto['product_id'].duplicated().sum())
duplicados_prod = dim_producto[dim_producto['product_id'].duplicated(keep=False)]
print(duplicados_prod.sort_values('product_id').head(20))

0
Empty DataFrame
Columns: [product_id, product_name, category, sub_category]
Index: []


Ya tenemos tres tablas de dimensiones limpias.

In [28]:
# Agregar location_id al dataframe mediante merge
df_con_location = df.merge(
    dim_ubicacion[['city', 'state', 'location_id']],
    left_on=['City', 'State'],
    right_on=['city', 'state'],
    how='left'
)

# Tabla de hechos con location_id
fact_ventas = df_con_location[[
    'Order ID',
    'Order Date',
    'Ship Date',
    'Ship Mode',
    'Customer ID',
    'Product ID',
    'location_id',
    'Sales',
    'Quantity',
    'Discount',
    'Profit',
    'Profit Margin %',
    'Days to Ship',
    'Order Year',
    'Order Month',
    'Order Quarter'
]].copy()

fact_ventas.columns = [
    'order_id',
    'order_date',
    'ship_date',
    'ship_mode',
    'customer_id',
    'product_id',
    'location_id',
    'sales',
    'quantity',
    'discount',
    'profit',
    'profit_margin_pct',
    'days_to_ship',
    'order_year',
    'order_month',
    'order_quarter'
]

print(f"Tabla de hechos: {len(fact_ventas)} registros")
print(fact_ventas.head())

Tabla de hechos: 22718 registros
         order_id order_date  ship_date     ship_mode customer_id  \
0  CA-2016-152156 2016-11-08 2016-11-11  Second Class    CG-12520   
1  CA-2016-152156 2016-11-08 2016-11-11  Second Class    CG-12520   
2  CA-2016-138688 2016-06-12 2016-06-16  Second Class    DV-13045   
3  CA-2016-138688 2016-06-12 2016-06-16  Second Class    DV-13045   
4  CA-2016-138688 2016-06-12 2016-06-16  Second Class    DV-13045   

        product_id  location_id   sales  quantity  discount    profit  \
0  FUR-BO-10001798            1  261.96         2       0.0   41.9136   
1  FUR-CH-10000454            1  731.94         3       0.0  219.5820   
2  OFF-LA-10000240            2   14.62         2       0.0    6.8714   
3  OFF-LA-10000240            4   14.62         2       0.0    6.8714   
4  OFF-LA-10000240           14   14.62         2       0.0    6.8714   

   profit_margin_pct  days_to_ship  order_year  order_month  order_quarter  
0               16.0             3  

In [29]:
dim_cliente.to_csv('../data/processed/dim_cliente.csv', index=False)
dim_producto.to_csv('../data/processed/dim_producto.csv', index=False)
dim_ubicacion.to_csv('../data/processed/dim_ubicacion.csv', index=False)
fact_ventas.to_csv('../data/processed/fact_ventas.csv', index=False)

print("Tablas exportadas correctamente:")
print("- dim_cliente.csv")
print("- dim_producto.csv")
print("- dim_ubicacion.csv")
print("- fact_ventas.csv")

Tablas exportadas correctamente:
- dim_cliente.csv
- dim_producto.csv
- dim_ubicacion.csv
- fact_ventas.csv


Lo que se construyo hasta acá:

-Carga y exploracion del dataset

-Convertir fechas de texto a datetime

-Crear columnas nuevas: Days to Ship, Profit Margin %, Order Year, Order Month, Order Quarter

-Separar el dato en 4 tablas limpias con estructura dimensional

-Exportar todo a data/processed/